# Model Application - CD20

This notebook shows the full pipeline of preprocessing, model training and metrics.

This dataset is simple and consists of a homogeneous cell line (Raji), with one sample being polarized with Rituximab (polarizes CD20). The polarization data used here are the precomputed PixelGen values.

The takeaway is that the model successfully jointly models abundance + polarization, achieving separation in the central CD20 polarization feature.

The best performing model seems to be the shared encoder. We model the (filtered & standardized) polarization with a normal distribution.

In [ ]:
import anndata
import pixelator
import torch
import scvi
import scipy
# from scvi import autotune

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

# import ray
# from ray import tune

from pathlib import Path


from PixelGen.pxl_utils import train_model, get_model_latents, convert_polarization_to_feature_matrix, \
    convert_colocalization_to_feature_matrix, download_pxl
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA

from pixelator.plot import molecule_rank_plot, cell_count_plot, scatter_umi_per_upia_vs_tau
from pixelator.statistics import clr_transformation
from pixelator.analysis.normalization import dsb_normalize


from sklearn.preprocessing import StandardScaler, MinMaxScaler 


import tempfile

from scvi import REGISTRY_KEYS
from scvi.module.base import (
    BaseModuleClass,
    LossOutput,
    PyroBaseModuleClass,
    auto_move_data,
)
from torch.distributions import NegativeBinomial, Normal, Poisson, MixtureSameFamily, Beta
from torch.distributions import kl_divergence as kl

from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D
from PixelGen.metrics import MultiModalVIMetrics

# from cytovi import CytoVI

print(torch.cuda.is_available())


scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
%load_ext autoreload
%autoreload 2

In [ ]:
DATA_DIR = Path('./PixelGen/datasets/cd20-rituximab')


FILENAMES = [
    "Sample03_Raji_control.layout.dataset.pxl",
    "Sample04_Raji_Rituximab_treated.layout.dataset.pxl",
]

SAMPLE_NAMES = [
    "control", 
    "treated",
]

COMBINED_FILENAME = "cd20_combined.pxl"
COMBINED_PATH = DATA_DIR / COMBINED_FILENAME

pg_data = pixelator.read(COMBINED_PATH)

# Uncomment to download for first time

# BASEURL = "https://pixelgen-technologies-datasets.s3.eu-north-1.amazonaws.com/mpx-datasets/pixelator/0.18.x/cd20-rituximab-v1.0-immunology-I"
# pg_data = download_pxl(
#     baseurl=BASEURL,
#     filenames=FILENAMES,
#     sample_names=SAMPLE_NAMES,
#     dataset_dir=DATA_DIR,
#     dataset_full_path=COMBINED_PATH,
# )

### Preprocessing and QC
(for documentation. Can skip to 'After Preprocessing' if not the first time reading notebook)

In [ ]:
adata = pg_data.adata.copy()
sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True, var_type='proteins')

adata.raw = adata.copy()
orig_adata = adata.copy()


In [ ]:
sc.pl.violin(
    adata,
    ["n_proteins_by_counts", "total_counts",],
    groupby='sample',
    jitter=0.3,
    multi_panel=True,
)

In [ ]:
molecule_rank_df = adata.obs[["sample", "molecules"]].copy()
molecule_rank_df["rank"] = molecule_rank_df.groupby(["sample"])["molecules"].rank(
    ascending=False, method="first"
)
fig_intersection, ax = molecule_rank_plot(molecule_rank_df, group_by="sample")
high = 80000
low = 20000
ax.axhline(y=high)
ax.axhline(y=low)

In [ ]:
cells_per_sample_df = (
    adata.obs.groupby("sample").size().to_frame(name="size").reset_index()
)

fig, ax = cell_count_plot(adata.obs, color_by="sample")

In [ ]:
tau_metrics_df = adata.obs[["sample", "tau", "mean_molecules_per_a_pixel", "tau_type"]]
tau_metrics_df = tau_metrics_df.rename(columns={"mean_molecules_per_a_pixel": "umi_per_upia"})


fig, ax = scatter_umi_per_upia_vs_tau(tau_metrics_df, group_by="sample")

In [ ]:
components = orig_adata.obs[
    (orig_adata.obs['tau_type'] == 'normal') & 
    (orig_adata.obs['molecules'] > low) & 
    (orig_adata.obs['molecules'] < high)
].index
vars = [v for v in adata.var_names]
adata = orig_adata[components, vars]
cells_per_sample_df = (
    adata.obs.groupby("sample").size().to_frame(name="size").reset_index()
)

fig, ax = cell_count_plot(adata.obs, color_by="sample")

## Ritu & CD20 Expression

Ritu masks CD20 in treated condition, explaining the low counts of CD20. Notice that the counts of Ritu (+ CD20) are higher than CD20 alone in the control condition - this is supposedly a due to difference in affinity.

In [ ]:
treated_components = [c for c in adata.obs.index if 'treated' in c]
control_components = [c for c in adata.obs.index if 'control' in c]
fig, ax = plt.subplots(1, 2)
sns.histplot({'control CD20': orig_adata.to_df().loc[control_components, 'CD20'], 'treated Ritu + CD20': orig_adata.to_df().loc[treated_components, 'Rituximab'] + orig_adata.to_df().loc[treated_components, 'CD20']}, log_scale=True, ax=ax[1])
sns.histplot({'control CD20': orig_adata.to_df().loc[control_components, 'CD20'], 'treated CD20': orig_adata.to_df().loc[treated_components, 'CD20']}, log_scale=True, ax=ax[0])

In [ ]:
ax = sc.pl.highest_expr_genes(adata, n_top=20, show=False)
ax.set_title('Highly Abundant Antibodies')
stats = adata.to_df().agg(['mean', 'var',], axis=0).T
fig, ax = plt.subplots(1)
sns.scatterplot(x=stats['mean'], y=stats['var'], ax=ax)
ax.loglog()
# var_genes = sc.pp.highly_variable_genes(adata, flavor='seurat_v3', n_top_genes=20, batch_key=batch_key, layer='counts', inplace=False)
# sc.pl.highly_variable_genes(var_genes, show=True, log=True)

In [ ]:
adata.layers['counts'] = adata.X.copy()
adata.layers['dsb'] = dsb_normalize(adata.to_df('counts'), isotype_controls=['mIgG1', 'mIgG2a', 'mIgG2b'])
adata.layers['clr_by_cell'] = clr_transformation(adata.to_df('counts'), axis=1)
adata.layers['clr_by_ab'] = clr_transformation(adata.to_df('counts'), axis=0)
adata.layers['log1p'] = np.log1p(adata.to_df('counts'))

In [ ]:
isotype_control = ['mIgG1', 'mIgG2a', 'mIgG2b']
pol_vars = [v for v in adata.var_names if v not in isotype_control]

polarization_i = convert_polarization_to_feature_matrix(pg_data.polarization, components=adata.obs.index, key='morans_i', vars=pol_vars)
polarization_z = convert_polarization_to_feature_matrix(pg_data.polarization, components=adata.obs.index, key='morans_z', vars=pol_vars)

adata.obsm['pol_i'] = polarization_i
adata.obsm['pol_z'] = polarization_z

adata.obsm['pol_i']['CD20_pol'] = adata.obsm['pol_i']['CD20_pol'].where(adata.obs['sample'] == 'control', other=adata.obsm['pol_i']['Rituximab_pol'])
adata.obsm['pol_i'].drop(columns='Rituximab_pol', inplace=True)
adata.obs['CD20_pol'] = adata.obsm['pol_i']['CD20_pol']

# Drop CD20 and Rituximab counts to not create artifical differences between the conditions
vars = [v for v in adata.var_names if v not in ['CD20', 'Rituximab'] + isotype_control]
adata = adata[:, vars]

sns.histplot(adata.obs, x='CD20_pol', hue='sample')

# Unit: in [0,1]
for key in ('pol_i',):
    adata.obsm[f'{key}_unit'] = (1 + adata.obsm[key]) / 2

# Standardization and variance filtering
pol = adata.obsm['pol_i_unit']
pol_std = (pol - pol.mean(axis=0)) / pol.std(axis=0)
pol_std_arcsinh = np.arcsinh(pol_std)
pol_hvg = pol.loc[:, pol.std(axis=0) > 0.01]
pol_hvg_std = (pol_hvg - pol_hvg.mean(axis=0)) / pol_hvg.std(axis=0)
pol_hvg_std_arcsinh = np.arcsinh(pol_hvg_std)

adata.obsm['pol_hvg'] = pol_hvg_std_arcsinh

In [ ]:
adata.obsm['abundance_pol'] = pd.concat(
    (adata.to_df('dsb'), adata.obsm['pol_hvg']),
    axis=1,
)
calc_PCA(adata, rep='abundance_pol', key_added='abundance_pol')
calc_PCA(adata, rep='dsb', key_added='dsb')

### PCA - Abundance Only vs Abundance + Pol
PCA of Abundance + Pol shows reasonable separation according to CD20 polarization

In [ ]:
_ = pca_neighbors_umap(adata, 'dsb', umap_pl_kwargs=dict(layer='dsb', color=['sample', 'CD38', 'CD20_pol'], vcenter=0, cmap='RdBu_r'), umap_title='Abundance Only')
_ = pca_neighbors_umap(adata, 'abundance_pol', umap_pl_kwargs=dict(layer='dsb', color=['sample', 'CD38', 'CD20_pol'], vcenter=0, cmap='RdBu_r'), umap_title='Abundance + Pol')

### After Preprocessing

In [ ]:
# adata.write_h5ad(DATA_DIR / 'cd20_filtered_with_pol.h5ad')
adata = anndata.read_h5ad(DATA_DIR / 'cd20_filtered_with_pol.h5ad')

## Abundance Model

In [ ]:
model_cls = MultiModalSCVI
setup_kwargs = dict(layer='dsb', batch_key=None)
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, plan_kwargs=dict(lr=3e-4, optimizer='Adam', n_epochs_kl_warmup=400))
model_kwargs = dict(n_latent=20, n_hidden=128, n_layers=1, dropout_rate=0.1, distrs=[D.Normal,],)
modalities_latent_names=[('dsb', 'abundance_model')]
abundance_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)
get_model_latents(adata, abundance_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['sample', 'CD38', 'CD20_pol',], vcenter=0, cmap='RdBu_r', layer='dsb')).suptitle(title)

## Polarization-only Model

In [ ]:
pol = adata.obsm['pol_i_unit']
pol_std = (pol - pol.mean(axis=0)) / pol.std(axis=0)
pol_std_arcsinh = np.arcsinh(pol_std)
pol_hvg = pol.loc[:, pol.std(axis=0) > 0.01]
pol_hvg_std = (pol_hvg - pol_hvg.mean(axis=0)) / pol_hvg.std(axis=0)
pol_hvg_std_arcsinh = np.arcsinh(pol_hvg_std)
adata.obsm['pol_hvg'] = pol_hvg_std_arcsinh


pol_adata = anndata.AnnData(
    X=adata.obsm['pol_i'],
    obs=adata.obs,
    layers={
        'pol_i': adata.obsm['pol_i'],
        'pol_i_unit': pol,
        'pol_i_std': pol_std,
        'pol_i_std_arcsinh': pol_std_arcsinh,
    }
)

pol_hvg_adata = anndata.AnnData(
    X=pol_hvg,
    obs=adata.obs,
    layers={
        'pol_i_hvg_std': pol_hvg_std,
        'pol_hvg': pol_hvg_std_arcsinh,
    }
)


try:
    pol_adata.obs.drop(columns=['CD20_pol'], inplace=True)
    pol_hvg_adata.obs.drop(columns=['CD20_pol'], inplace=True)
except:
    pass

adata.obs['CD20_pol_arcsinh'] = pol_hvg_std_arcsinh['CD20_pol']

In [ ]:
model_cls = MultiModalSCVI

cur_adata = pol_hvg_adata
layer = 'pol_hvg'

cur_adata.obs['CD20_pol_obs'] = cur_adata.to_df(layer)['CD20_pol']

setup_kwargs = dict(layer=layer, extra_modality_keys=[], n_modalities=1, batch_key=None, )
model_kwargs = dict(n_latent=20, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal], 
                        joint_kl=False, unimodal_kl=True,
                        external_kl_weight=1, decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp'),
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, plan_kwargs=dict(lr=3e-4, optimizer='Adam', n_epochs_kl_warmup=400))

latent_name = 'pol_model'
modalities_latent_names=[(layer, latent_name)]
pol_model = train_model(cur_adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs)
get_model_latents(cur_adata, pol_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(cur_adata, name, umap_pl_kwargs=dict(color=['sample', 'CD20_pol_obs',], vcenter=0, cmap='RdBu_r')).suptitle(title)

## Abundance + Pol Model

### Separate Encoders, Learned Global Weights
This type of model learns separate latents for each modality, allowing one to view and cluster according to each separate modality as well as the joint latent. This comes at varying cost to performance, due to the inherent limitation on the inference of the joint latent.

In [ ]:
model_cls = MultiModalSCVI

pol_key = 'pol_hvg'
latent_name = 'abundance_pol_model'

setup_kwargs = dict(layer='dsb', extra_modality_keys=[pol_key], n_modalities=2, batch_key=None, )
model_kwargs = dict(n_latent=20, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal, D.Normal], 
                        agg_method=AggMethod.AOE_GLOBAL_WEIGHTS,
                        joint_kl=False, unimodal_kl=True,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp'),
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, plan_kwargs=dict(lr=3e-4, optimizer='Adam', n_epochs_kl_warmup=400))
modalities_latent_names=[('joint', latent_name), ('dsb', f'{latent_name}_dsb'), (pol_key, f'{latent_name}_{pol_key}')]
abundance_pol_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)
weights = abundance_pol_model.get_weights()
get_model_latents(adata, abundance_pol_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name if key != 'joint' else f'{name}, weights: {np.array2string(weights, precision=2)}'    
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['sample', 'CD38', 'CD20_pol',], vcenter=0, cmap='RdBu_r', layer='dsb')).suptitle(title)

### Shared Encoder

In [ ]:
model_cls = MultiModalSCVI

pol_key = 'pol_hvg'
latent_name = 'abundance_pol_shared_enc_model'

setup_kwargs = dict(layer='dsb', extra_modality_keys=[pol_key], n_modalities=2, batch_key=None, )
model_kwargs = dict(n_latent=20, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal, D.Normal], 
                        agg_method=AggMethod.SHARED_ENCODER,
                        joint_kl=True, unimodal_kl=False,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, plan_kwargs=dict(lr=3e-4, optimizer='Adam', n_epochs_kl_warmup=400))
abundance_pol_shared_enc_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

modalities_latent_names=[('joint', latent_name)]
get_model_latents(adata, abundance_pol_shared_enc_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['sample', 'CD38', 'CD20_pol',], vcenter=0, cmap='RdBu_r', layer='dsb')).suptitle(title)

### Shared Encoder - Using batch key
Here the decoder gets the batch key, so it should be much easier to decode the CD20 polarization, hence separation in the latent space is unnecessary (essentially this is batch integration where we treat the CD20 polarization as a batch effect)

In [ ]:
model_cls = MultiModalSCVI

pol_key = 'pol_hvg'
latent_name = 'abundance_pol_shared_enc_batch_model'

setup_kwargs = dict(layer='dsb', extra_modality_keys=[pol_key], n_modalities=2, batch_key='sample', )
model_kwargs = dict(n_latent=20, n_hidden=128, n_layers=1, dropout_rate=0.1, 
                        distrs=[D.Normal, D.Normal], 
                        agg_method=AggMethod.SHARED_ENCODER,
                        joint_kl=True, unimodal_kl=False,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, early_stopping_monitor='elbo_validation',
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, plan_kwargs=dict(lr=1e-3, optimizer='Adam', n_epochs_kl_warmup=400))
modalities_latent_names=[('joint', latent_name),]
shared_enc_w_batch_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs, 
                    modalities_latent_names=modalities_latent_names)
get_model_latents(adata, shared_enc_w_batch_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['sample', 'CD38', 'CD20_pol',], vcenter=0, cmap='RdBu_r', layer='dsb')).suptitle(title)

## Metrics

In [ ]:
metrics = MultiModalVIMetrics(
    adata,
    models = {
        'abundance_only': abundance_model,
        'global_weights': abundance_pol_model,
        'shared_enc': abundance_pol_shared_enc_model,
        'shared_enc_w_batch': shared_enc_w_batch_model,
        'pol_model': pol_model,
    },
    pca_key='abundance_pol_pca'
)
metrics.run()

Mean modality errors: notice the difference between stochastic reconstruction (i.e. sampling from the distribution according to the predicted parameters) or just taking the mean. The latter has lower MSE by definition, because the former has added variance. Hence the error of the model when using a stochastic reconstruction can be comparable or even higher than the error of the mean baseline (on a unimodal distribution). However, only when using a stochastic reconstruction can one compare the reconstruction distribution to the ground truth.

Models achieve similar errors for dsb (when reconstructing stochastically, error is similar to mean error, since the sample is homogenous cell-type-wise hence marker distributions are mostly unimodal).

Shared encoder & pol-only model achieve similar errors for polarization.
Note the global weights does slightly worse.

In [ ]:
_ = metrics.mean_modality_errors_barplot()
_ = metrics.mean_modality_errors_barplot(reconstruction_mean=True)

For autocorrelation in the latent space, notice pol-only and abundance-only latents behave as expected w.r.t autocorr of dsb and pol.

The low overall autocorr of dsb even in the dsb model can be attributed again to the fact that the sample is homogenous, therefore signifcant clustering according to marker distributions is not expected.

In [ ]:
_ = metrics.mean_autocorr_barplot()

Notice low autocorr of CD20_pol in the shared_enc_w_batch latent, due to what we said above.

In [ ]:
fig = metrics.autocorr_barplot(autocorr_key='pol_hvg',)
fig = metrics.autocorr_barplot(autocorr_key='dsb', auto_filter_features=15)
fig = metrics.top_autocorr_features_barplot(key='pol_hvg', top=10)
# fig.axes[0].legend_ = None

Notice nice prediction of CD20_pol distributions.

Slightly bad behavior of the pol model might be due to unstable training as also seen in the training curve (small number of features)

In [ ]:
fig = metrics.feature_histplot(modality='pol_hvg', features=['CD20_pol', 'CD38_pol'], hue='sample')
fig = metrics.feature_histplot(modality='dsb', features=['HLA-DR', 'CD38'], hue='sample')

More detailed error plots

In [ ]:
fig = metrics.errors_barplot(modality='dsb', auto_filter_features=15, reconstruction_mean=False)
fig = metrics.errors_barplot(modality='dsb', auto_filter_features=15, reconstruction_mean=True, skip_mean_baseline=True)
fig = metrics.errors_barplot(modality='pol_hvg', reconstruction_mean=True)

### Altogether the metrics seem to find that the shared encoder model excels.